# Plant–arthropod benchmark: Python quick start

This notebook imports the public Parquet tables, selects the provisional direct E0/E1 layer, summarizes the usable networks, and constructs one shared-host projection. It is deliberately small enough to adapt for a thesis analysis.

> **Preview warning:** human ecological review is still in progress. A projected edge means shared-host overlap, not demonstrated competition.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'data' / 'parquet' / 'networks.parquet').is_file()
)
DATA = ROOT / 'data' / 'parquet'
str(DATA.relative_to(ROOT))

'data/parquet'

## 1. Import the core tables

`networks` is the selection table. `interactions` is the canonical long plant-by-herbivore table. `taxa` retains source-local names, and `network_metrics` contains analysis gates and graph summaries.

In [2]:
networks = pd.read_parquet(DATA / 'networks.parquet')
interactions = pd.read_parquet(DATA / 'interactions.parquet')
taxa = pd.read_parquet(DATA / 'taxa.parquet')
metrics = pd.read_parquet(DATA / 'network_metrics.parquet')

pd.Series({
    'network objects': len(networks),
    'interaction cells': len(interactions),
    'observed positive links': (interactions['interaction_state'] == 'observed_positive').sum(),
    'source-local taxon records': len(taxa),
}, name='count').to_frame()

,count
network objects,346
interaction cells,848907
observed positive links,19855
source-local taxon records,33812


## 2. Select a defensible analysis layer

For a first direct-data analysis, retain provisional E0/E1 networks and use `independence_cluster_id` to avoid treating nested or repeated observations as independent replicates. An analysis gate is a transparent size threshold, not a quality score.

In [3]:
direct = networks.loc[
    networks['eligible_direct']
    & networks['evidence_class'].isin(['E0', 'E1'])
].copy()
comparative = direct.loc[direct['gate_comparative']].copy()

pd.Series({
    'provisional direct networks': len(direct),
    'direct independence clusters': direct['independence_cluster_id'].nunique(),
    'direct original studies': direct['study_id'].nunique(),
    'networks passing comparative gate': len(comparative),
}, name='count').to_frame()

,count
provisional direct networks,126
direct independence clusters,38
direct original studies,8
networks passing comparative gate,68


In [4]:
study_summary = (
    comparative.groupby('study_id')
    .agg(
        networks=('network_id', 'size'),
        independence_clusters=('independence_cluster_id', 'nunique'),
        median_plants=('active_plant_count', 'median'),
        median_herbivores=('active_herbivore_count', 'median'),
        median_positive_links=('positive_links', 'median'),
    )
    .sort_values(['networks', 'independence_clusters'], ascending=False)
)
study_summary

,networks,independence_clusters,median_plants,median_herbivores,median_positive_links
study_id,,,,,
saavedra_2018_succession,33,9,18.0,29.0,43.0
martins_2020_cerrado,11,10,12.0,28.0,51.0
redmond_2018_succession,9,9,25.0,87.0,131.0
volf_2017_temperate,8,3,8.0,49.0,53.5
seifert_2021_temperate,3,3,15.0,141.0,399.0
pocock_2012_norwood,2,1,18.0,23.5,29.5
shinohara_2020_grassland,1,1,11.0,11.0,20.0
sousa_lopes_2024_cerrado,1,1,5.0,87.0,94.0


## 3. Build one plant × herbivore incidence matrix

The example chooses the largest qualifying direct network by positive-link count. It uses only `observed_positive` records. Zeros in this derived incidence view mean that no positive edge is represented; they do not turn unknown or unsampled source combinations into biological absences.

In [5]:
network_id = (
    comparative.sort_values(['positive_links', 'network_id'], ascending=[False, True])
    .iloc[0]['network_id']
)
positive = interactions.loc[
    (interactions['network_id'] == network_id)
    & (interactions['interaction_state'] == 'observed_positive')
].copy()

B = positive.pivot_table(
    index='plant_taxon_record_id',
    columns='herbivore_taxon_record_id',
    values='binary_value',
    aggfunc='max',
    fill_value=0,
).astype('int64')

network_id, B.shape, len(positive)

('seifert_2021_tomakomai_japan', (20, 148), 718)

## 4. Compute the potential-competition graph

For binary incidence `B`, `C = B.T @ B`. The diagonal is herbivore host breadth. Positive off-diagonal cells count shared plant hosts between herbivore pairs.

In [6]:
C = B.T @ B
host_breadth = B.sum(axis=0).to_numpy()
assert np.array_equal(np.diag(C.to_numpy()), host_breadth)

off_diagonal = C.to_numpy(copy=True)
np.fill_diagonal(off_diagonal, 0)
projected_edge_count = int(np.count_nonzero(np.triu(off_diagonal, k=1)))

pd.Series({
    'network_id': network_id,
    'active plants': B.shape[0],
    'active herbivores': B.shape[1],
    'positive bipartite links': len(positive),
    'positive shared-host edges': projected_edge_count,
    'minimum host breadth': int(host_breadth.min()),
    'mean host breadth': float(host_breadth.mean()),
    'maximum host breadth': int(host_breadth.max()),
}, name='value').to_frame()

,value
network_id,seifert_2021_tomakomai_japan
active plants,20
active herbivores,148
positive bipartite links,718
positive shared-host edges,7553
minimum host breadth,1
mean host breadth,4.851351
maximum host breadth,19


In [7]:
name_map = (
    taxa.loc[taxa['network_id'] == network_id]
    .set_index('taxon_record_id')['name_verbatim']
)
upper = np.triu_indices_from(off_diagonal, k=1)
top_pairs = pd.DataFrame({
    'herbivore_a_id': C.index.to_numpy()[upper[0]],
    'herbivore_b_id': C.columns.to_numpy()[upper[1]],
    'shared_host_count': off_diagonal[upper],
}).query('shared_host_count > 0')
top_pairs['herbivore_a'] = top_pairs['herbivore_a_id'].map(name_map)
top_pairs['herbivore_b'] = top_pairs['herbivore_b_id'].map(name_map)
top_pairs.sort_values('shared_host_count', ascending=False)[
    ['herbivore_a', 'herbivore_b', 'shared_host_count']
].head(10)

,herbivore_a,herbivore_b,shared_host_count
9403,Lymantria_mathura,Operophtera_brunnea,18
9238,Lomographa_simplicior,Operophtera_brunnea,16
6829,Erannis_golda,Operophtera_brunnea,16
5672,Cosmia_trapezina,Lymantria_mathura,16
6818,Erannis_golda,Lymantria_mathura,16
9227,Lomographa_simplicior,Lymantria_mathura,16
5669,Cosmia_trapezina,Lomographa_simplicior,15
5258,Conistra_sp_3,Lymantria_mathura,15
5683,Cosmia_trapezina,Operophtera_brunnea,15
5269,Conistra_sp_3,Operophtera_brunnea,15


## Next steps

Before a comparative analysis, read `docs/ANALYSIS_NOTES.md`, choose a declared evidence layer and analysis gate, retain study and independence-cluster identifiers, and repeat key summaries under nearby thresholds. Cite both this collection version and the relevant original source datasets.